# Solving Linear Systems and Conditioning

This notebook reuses `solve` from `linalg_with_python` to tackle a few linear systems, inspect residuals, and highlight how conditioning influences the stability of solutions.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from linalg_with_python.systems import solve

FIGURES_DIR = Path.cwd() / "assets" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


## Solving canonical systems

We solve a simple 2×2 system, inspect the residual, and then extend to a 3×3 example to verify the solution quality.


In [ ]:

A = np.array([[3.0, 1.0], [1.0, 2.0]], dtype=np.float64)
b = np.array([9.0, 7.0], dtype=np.float64)
result = solve(A, b)
residual = A @ result.x - b
print(f"Solution: {result.x}")
print(f"Residual norm: {np.linalg.norm(residual):.3e}")

A3 = np.array([[2.0, -1.0, 0.0], [0.0, 3.0, 1.0], [1.0, 2.0, 4.0]])
b3 = np.array([2.0, 5.0, 9.0], dtype=np.float64)
result3 = solve(A3, b3)
print(f"3×3 solution: {result3.x}")
print(f"3×3 residual norm: {np.linalg.norm(A3 @ result3.x - b3):.3e}")


## Residuals and geometry

The vector $Ax - b$ is the residual. A small residual says the point $x$ lies close to the affine plane defined by $Ax = b$.


## Conditioning and sensitivity

Compare a well-conditioned matrix with a nearly singular one, then perturb the right-hand side to expose the amplification of errors.


In [ ]:

well = np.array([[4.0, 1.0], [1.0, 3.0]], dtype=np.float64)
bad = np.array([[1.0, 1.0], [1.0, 1.0 + 1e-8]], dtype=np.float64)
for label, matrix in [("Well-conditioned", well), ("Ill-conditioned", bad)]:
    cond = np.linalg.cond(matrix)
    sol = np.linalg.solve(matrix, np.array([2.0, 1.0]))
    print(f"{label}: cond={cond:.3e}, solution={sol}")


In [ ]:

base = np.array([2.0, 1.0], dtype=np.float64)
perturb = np.array([1e-6, -1e-6], dtype=np.float64)
sol_good = np.linalg.solve(well, base + perturb)
sol_bad = np.linalg.solve(bad, base + perturb)
sol_good_base = np.linalg.solve(well, base)
sol_bad_base = np.linalg.solve(bad, base)
print(f"Well-conditioned change: {np.linalg.norm(sol_good - sol_good_base):.3e}")
print(f"Ill-conditioned change: {np.linalg.norm(sol_bad - sol_bad_base):.3e}")


In [ ]:

matrix = bad
base_vec = np.array([2.0, 1.0], dtype=np.float64)
perturbations = np.geomspace(1e-12, 1e-2, 9)
changes = []
base_sol = np.linalg.solve(matrix, base_vec)
for eps in perturbations:
    perturbed = base_vec + np.array([eps, -eps])
    sol = np.linalg.solve(matrix, perturbed)
    changes.append(np.linalg.norm(sol - base_sol))

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(perturbations, changes, marker='o')
ax.set_title('Sensitivity of solution to perturbations')
ax.set_xlabel('Perturbation magnitude (log scale)')
ax.set_ylabel('||Δx||₂')
ax.grid(True, which='both', linestyle='--', linewidth=0.5)
path = FIGURES_DIR / 'system_solution_sensitivity.png'
fig.savefig(path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved sensitivity figure to {path}")


In [ ]:
from linalg_with_python.checks import assert_close
from linalg_with_python.systems import solve
import numpy as np
A = np.array([[3.0, 1.0], [1.0, 2.0]], dtype=np.float64)
b = np.array([9.0, 7.0], dtype=np.float64)
res = solve(A, b)
residual = A @ res.x - b
assert_close('system residual', residual, np.zeros_like(residual), tol=1e-12)
print('Residual is near zero as expected.')

### Conditioning laboratory

A well-conditioned matrix keeps solutions stable while an ill-conditioned one amplifies tiny noise. Run `python scripts/demo_conditioning.py` to regenerate the figure, then inspect the following comparison.


In [ ]:
import numpy as np
from IPython.display import Image

A_good = np.array([[4.0, 1.0], [1.0, 3.0]], dtype=np.float64)
A_bad = np.array([[1.0, 1.0], [1.0, 1.0 + 1e-8]], dtype=np.float64)
print(f"cond(A_good)={np.linalg.cond(A_good):.3e}")
print(f"cond(A_bad)={np.linalg.cond(A_bad):.3e}")

print("Figure illustrates relative error vs perturbation magnitude:")
Image("assets/figures/conditioning_sensitivity.png")